<div style="background-color: #000000; padding: 10px; border-radius: 5px;">
<img src="images/logo.png" width="100">

In [31]:
# find the center of the ligand
import MDAnalysis as mda
original_structure = mda.Universe("protein_structures/chain_A.pdb")
ligand_mda = original_structure.select_atoms("resname CP6 and resid 609")
# Get the center of the ligand as the "pocket center"
pocket_center = ligand_mda.center_of_geometry()## Fill in code for measuring the center of geometry of the ligand.
print(pocket_center)

[28.03476468  5.43635293 58.75600007]


In [32]:
from vina import Vina
v = Vina(sf_name="vina")

In [33]:
#! grep " A  86 " protein_structures/chain_A_cleanH.pdb

In [34]:
!mk_prepare_receptor.py \
-i protein_structures/chain_A_cleanH.pdb \
-o protein_structures/chain_A_clean \
 -p

@> 9042 atoms and 1 coordinate set(s) were parsed in 0.09s.

Files written:
protein_structures/chain_A_clean.pdbqt <-- static (i.e., rigid) receptor input file


In [35]:
from vina import Vina
from pathlib import Path
import os
ligand_dir = Path("pdbqt")
out_dir = Path("docking results")
out_dir.mkdir(exist_ok=True)
for ligand in ligand_dir.glob("*.pdbqt"):
    v = Vina(sf_name="vina", seed=42)
    v.set_receptor("protein_structures/chain_A_clean.pdbqt")
    v.set_ligand_from_file(str(ligand))
    v.compute_vina_maps(
        center=pocket_center,
        box_size=[20, 20, 20]
    )
    v.dock(exhaustiveness=8, n_poses=5)
    print(f"\n{ligand.name}")
    for i, e in enumerate(v.energies(), 1):
        print(f"Pose {i}: {e[0]:.2f} kcal/mol")
    v.write_poses(
        str(out_dir / f"{ligand.stem}_poses.pdbqt"),
        n_poses=5,
        overwrite=True
    )


mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -6.715          0          0
   2       -6.605      4.239      6.517
   3       -6.507      2.259      3.257
   4       -6.463      3.644      5.159
   5       -6.453      1.629      2.125
Computing Vina grid ... done.
Performing docking (random seed: 42) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

pyrimethamine.pdbqt
Pose 1: -7.47 kcal/mol
Pose 2: -7.05 kcal/mol
Pose 3: -6.85 kcal/mol
Pose 4: -6.84 kcal/mol
Pose 5: -6.76 kcal/mol

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -7.473          0          0
   2       -7.049      7.223      9.303
   3       -6.854      4.343      6.613
   4       -6.839      4.058      5.244
   5       -6.756      3.99

TypeError: 

PDBQT parsing error: Unknown or inappropriate tag found in flex residue or ligand.
 > TITLE     FRAME 0 FROM protein_structures/chain_A_clean.pqr


In [ ]:
#convert pdbqt dock poses into SDF
import glob
import os

for pdbqt in glob.glob("docking results/*.pdbqt"):
    sdf = os.path.splitext(pdbqt)[0] + ".sdf"
    !mk_export.py "$pdbqt" -s "$sdf"